# Customer Segmentation with an AI Coding Assistant

This notebook is a guided interface to the reproducible experiment in `src/run_experiment.py`. It compares K-Means, Ward hierarchical clustering, and DBSCAN on standardized age, income, and spending features.

## 1. Setup and dataset audit

Place `Mall_Customers.csv` in `data/` before running. CustomerID and Gender are audited but excluded from the clustering feature matrix.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'Mall_Customers.csv'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'

df = pd.read_csv(DATA_PATH)
display(df.head())
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print('Duplicate customer IDs:', int(df['CustomerID'].duplicated().sum()))

## 2. Execute the reproducible experiment

The command below regenerates every metric, customer assignment, profile, and figure.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'src' / 'run_experiment.py'),
    '--data', str(DATA_PATH),
    '--output', str(OUTPUT_DIR),
], check=True)

## 3. Model comparison

DBSCAN's silhouette is computed on non-noise observations, so coverage must be considered beside it.

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'model-comparison.csv')
display(comparison.sort_values('coverage_adjusted_silhouette', ascending=False).head(12))
display(Image(filename=str(OUTPUT_DIR / 'model-selection.png')))

## 4. Selected customer segments

Six-cluster K-Means is selected because it has the strongest full-coverage silhouette score and assigns every customer. Persona names summarize cluster averages and are not causal or psychological labels.

In [ ]:
profiles = pd.read_csv(OUTPUT_DIR / 'cluster-profiles.csv')
display(profiles)
display(Image(filename=str(OUTPUT_DIR / 'customer-segments.png')))
display(Image(filename=str(OUTPUT_DIR / 'cluster-personas.png')))

## 5. Verified summary and limitations

Internal metrics quantify geometric separation, not business value. External validation, stability tests, campaign outcomes, and drift monitoring would be required before deployment.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'results-summary.json').read_text())
summary